In [1]:
import numpy as np
import pandas as pd

In [2]:
%pip -q install finlab TA-Lib


In [ ]:
"""Colab cell: API-key authentication and 2330 indicator checks."""
import contextlib, io, os, json, zipfile
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import finlab
from finlab import data
from IPython.display import display

# Key is never written into the notebook, report, or exported files.
_api_key = getpass('FinLab API Key（隱藏輸入）: ').strip()
if not _api_key:
    raise RuntimeError('尚未輸入 API Key')
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        finlab.login(_api_key)
except Exception:
    raise RuntimeError('API Key 登入失敗，請確認 Key 及帳戶權限') from None

In [ ]:
# 取得全部指數收盤點數
index_close = pd.DataFrame(
    data.get("stock_index_price:收盤指數")
)

# 取得上市產業價格指數
index_names = index_close.columns[
    index_close.columns.str.match(r"^上市.+類指數$")
]
industry_close = index_close[index_names]

# 取得上市大盤價格指數
benchmark_close = index_close["上市發行量加權股價指數"]


In [ ]:
# 日期由舊到新排序
industry_close = industry_close.sort_index()
benchmark_close = benchmark_close.sort_index()

# 使用產業與大盤共有的日期
dates = industry_close.index.intersection(benchmark_close.index)
industry_prices = industry_close.loc[dates]
benchmark_prices = benchmark_close.loc[dates]

# 保留大盤點數有效且大於零的日期
valid_dates = benchmark_prices.notna() & (benchmark_prices > 0)
industry_prices = industry_prices.loc[valid_dates]
benchmark_prices = benchmark_prices.loc[valid_dates]

# 產業點數為零或負值時，改為缺值，不補值
industry_prices = industry_prices.where(industry_prices > 0)

In [ ]:
def calculate_rs(industry_prices, benchmark_prices):
    """每日 RS = 產業收盤點數 ÷ 同日大盤收盤點數。"""
    return industry_prices.div(benchmark_prices, axis=0)


# 產生每日 RS，供後續計算使用
industry_rs = calculate_rs(industry_prices, benchmark_prices)

In [ ]:
# 計算近5日變化需有今日與5個交易日前的點數，共至少6筆
if len(industry_rs) < 6:
    raise ValueError("資料不足6筆，無法計算近5日 RS 變化")

# 當日漲跌幅（%）=（今日產業收盤點數 ÷ 前一交易日產業收盤點數 − 1）× 100
# (P[t] / P[t-1] - 1) × 100；這是產業自身漲跌，尚未與大盤比較
industry_return_1d = (
    industry_prices.iloc[-1] / industry_prices.iloc[-2] - 1
) * 100

# 近1日 RS 變化（%）=（今日 RS ÷ 前一交易日 RS − 1）× 100

rs_change_1d = (
    industry_rs.iloc[-1] / industry_rs.iloc[-2] - 1
) * 100

# 近3日 RS 變化（%）=（今日 RS ÷ 3個交易日前 RS − 1）× 100

rs_change_3d = (
    industry_rs.iloc[-1] / industry_rs.iloc[-4] - 1
) * 100

# 近5日 RS 變化（%）=（今日 RS ÷ 5個交易日前 RS − 1）× 100

rs_change_5d = (
    industry_rs.iloc[-1] / industry_rs.iloc[-6] - 1
) * 100

# 昨日的近3日 RS 變化（%）=（昨日 RS ÷ 4個交易日前 RS − 1）× 100

rs_change_3d_previous = (
    industry_rs.iloc[-2] / industry_rs.iloc[-5] - 1
) * 100

# 近3日 RS 變化較昨日增減（百分點）
# = 今日的近3日 RS 變化率 − 昨日的近3日 RS 變化率

rs_momentum_change = rs_change_3d - rs_change_3d_previous

# 整理最新五項指標；缺值保留 NaN
rs_summary = pd.DataFrame({
    "當日漲跌幅（%）": industry_return_1d,
    "近1日RS變化（%）": rs_change_1d,
    "近3日RS變化（%）": rs_change_3d,
    "近5日RS變化（%）": rs_change_5d,
    "近3日RS變化較昨日增減（百分點）": rs_momentum_change,
})

# 顯示最新資料日期
print(f"資料日期：{industry_prices.index[-1].date()}")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(rs_summary.round(2))
